In [1]:
!pip install tensorflow==2.10
!pip install image-classifiers==0.2.2
!pip install segmentation-models
!pip install psutil

ERROR: Could not find a version that satisfies the requirement tensorflow==2.10 (from versions: 2.12.0rc0, 2.12.0rc1, 2.12.0, 2.12.1, 2.13.0rc0, 2.13.0rc1, 2.13.0rc2, 2.13.0, 2.13.1, 2.14.0rc0, 2.14.0rc1, 2.14.0, 2.14.1, 2.15.0rc0, 2.15.0rc1, 2.15.0, 2.15.1, 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0)
ERROR: No matching distribution found for tensorflow==2.10


  Using cached image_classifiers-0.2.2-py2.py3-none-any.whl.metadata (8.3 kB)
Using cached image_classifiers-0.2.2-py2.py3-none-any.whl (72 kB)
  Attempting uninstall: image-classifiers
    Found existing installation: image-classifiers 1.0.0
    Uninstalling image-classifiers-1.0.0:
      Successfully uninstalled image-classifiers-1.0.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
segmentation-models 1.0.1 requires image-classifiers==1.0.0, but you have image-classifiers 0.2.2 which is incompatible.


  Using cached image_classifiers-1.0.0-py3-none-any.whl.metadata (8.6 kB)
Using cached image_classifiers-1.0.0-py3-none-any.whl (19 kB)
  Attempting uninstall: image-classifiers
    Found existing installation: image-classifiers 0.2.2
    Uninstalling image-classifiers-0.2.2:
      Successfully uninstalled image-classifiers-0.2.2


In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

In [ ]:
# Pfad zu Ihrem Projektordner in Google Drive
project_path = '/content/drive/MyDrive/sisa-fork'

# Arbeitsverzeichnis wechseln
os.chdir(project_path)

print("Aktuelles Verzeichnis:", os.getcwd())

In [2]:

import json
import csv
import time
import psutil
from datetime import datetime
from classification_models.keras import Classifiers
from tensorflow import keras
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (
    CSVLogger,
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
)
import subprocess
import sys
sys.path.append("C:/Users/sophi/OneDrive/Documents/TH/Bachelorarbeit/Code/notebooks/SISA/sisa-fork")
from datasets.CIFAR10.dataloader_all_data import load
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

C:\Users\sophi\miniconda3\envs\thesis\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


x_train shape: (45000, 32, 32, 3)
y_train shape: (45000, 1)
x_val shape: (5000, 32, 32, 3)
y_val shape: (5000, 1)
x_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000


TODO:
- Model laden + die Pytorch Impl anpassen -> DONE
- Datensaetze vorbereiten -> DONE
- Korrekte Argumente für Skripte -> DONE
- Validationdatensatz in Training, pruefen wie genutzt
- Extra Eval fuer Vergleichbarkeit
- Ausfuerhung in Colab moeglich machen -> Google Drive
- Datei für Unlearning nutzen


In [106]:
#Global Config
BATCH_SIZE = 64 #more balance for gradient updates
EPOCHS = 30
LEARNING_RATE = 0.001
IMAGE_SIZE = 32 # CIFAR-10 is 32x32, better resolution than with 64x64 ( no resizing noise)
NUM_CLASSES = 10

#initializes NumpY and Tenserflow to work deterministicaly
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

DATAFILEPATH= r"datasets/CIFAR10/datasetfile_all_data"
DATALOADER= "datasets.CIFAR10.dataloader_all_data"

DATAFILEPATH_UNLEARNING= r"datasets/CIFAR10/datasetfile_random10_deletion"
OPTIMIZER = "adam" # --optimizer adam

#SISA Specific
CONTAINER_NAME= "Baseline"
NUM_SHARDS = 10
NUM_SLICES = 5
MODEL_NAME="resnet18"
LABEL="0"

In [107]:
#Log what GPU is used in training -> Colab specific
!nvidia-smi

Der Befehl "nvidia-smi" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


In [108]:
#Resource Monitor for Efficiency Metrics

class ResourceMonitor(tf.keras.callbacks.Callback):

    def __init__(self):
        super().__init__()

        self.total_training_time = 0

        self.max_ram_gb = 0

        self.gpu_available = False
        self.max_gpu_gb = 0

        self.num_parameters = 0

        # History for plotting usage
        self.ram_history = []
        self.gpu_history = []
        self.epoch_history = []

    def on_train_begin(self, logs=None):

        self.train_start = time.perf_counter()

        self.num_parameters = self.model.count_params()

        gpus = tf.config.list_physical_devices("GPU")

        self.gpu_available = len(gpus) > 0

        if self.gpu_available:
            try:
                tf.config.experimental.reset_memory_stats(
                    "GPU:0"
                )
            except Exception:
                pass

    def on_epoch_end(self, epoch, logs=None):

        # RAM usage
        ram_gb = (
            psutil.Process(os.getpid())
            .memory_info()
            .rss
            / (1024 ** 3)
        )

        self.ram_history.append(ram_gb)

        self.max_ram_gb = max(
            self.max_ram_gb,
            ram_gb
        )

        gpu_gb = 0

        if self.gpu_available:
            try:
                gpu_info = tf.config.experimental.get_memory_info("GPU:0")

                gpu_gb = gpu_info["peak"] / (1024 ** 3)

                self.max_gpu_gb = max(
                    self.max_gpu_gb,
                    gpu_gb
                )

            except Exception:
                gpu_gb = 0

        self.gpu_history.append(gpu_gb)
        self.epoch_history.append(epoch + 1)

    def on_train_end(self, logs=None):

        self.total_training_time = (
            time.perf_counter()
            - self.train_start
        )

    def get_report(self):

        return {

            "training_time_sec":
                round(
                    self.total_training_time,
                    2
                ),

            "peak_ram_gb":
                round(
                    self.max_ram_gb,
                    2
                ),

            "gpu_used":
                self.gpu_available,

            "peak_gpu_gb":
                round(
                    self.max_gpu_gb,
                    2
                ),

            "num_parameters":
                int(
                    self.num_parameters
                )
        }

    def plot_usage(self):

        # RAM plot
        plt.figure(figsize=(8, 4))
        plt.plot(
            self.epoch_history,
            self.ram_history,
            marker="o"
        )
        plt.xlabel("Epoch")
        plt.ylabel("RAM Usage (GB)")
        plt.title("RAM Usage During Training")
        plt.grid(True)
        plt.show()

        # GPU plot
        if self.gpu_available:
            plt.figure(figsize=(8, 4))
            plt.plot(
                self.epoch_history,
                self.gpu_history,
                marker="o"
            )
            plt.xlabel("Epoch")
            plt.ylabel("GPU Usage (GB)")
            plt.title("GPU Usage During Training")
            plt.grid(True)
            plt.show()


In [109]:
def evaluate_model(model, test_ds, resource_monitor, model_name="model"):
    predictions = model.predict(test_ds, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)

    y_true = np.concatenate([y for _, y in test_ds], axis=0).flatten()

    accuracy = accuracy_score(y_true, predicted_classes)
    precision = precision_score(
        y_true,
        predicted_classes,
        average="weighted"
    )
    recall = recall_score(
        y_true,
        predicted_classes,
        average="weighted"
    )
    f1 = f1_score(
        y_true,
        predicted_classes,
        average="weighted"
    )
    cm = confusion_matrix(
        y_true,
        predicted_classes
    )

    print(f"Evaluation for {model_name}")
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)

    print("Classification Report")
    print(
        classification_report(
            y_true,
            predicted_classes
        )
    )

    return {
        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "confusion_matrix": cm
    }

In [110]:
# Daten laden (aus Baseline-Dataloader)
#Preproccesing already done in dataloader
from datasets.CIFAR10.dataloader_all_data import x_train, y_train, x_val, y_val, x_test, y_test

In [111]:
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

print("x_val shape:", x_val.shape)
print("y_val shape:", y_val.shape)

print("x_test shape:", x_test.shape)
print("y_test shape:", y_test.shape)

print("Number of training datapoints:", x_train.shape[0])
print("Number of validation datapoints:", x_val.shape[0])
print("Number of test datapoints:", x_test.shape[0])

x_train shape: (45000, 3, 32, 32)
y_train shape: (45000,)
x_val shape: (5000, 3, 32, 32)
y_val shape: (5000,)
x_test shape: (10000, 3, 32, 32)
y_test shape: (10000,)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000


In [112]:
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(50000, seed=SEED, reshuffle_each_iteration=False).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds=tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds =val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_ds =test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [113]:
print("\n=== TRAIN DATASET after Prepeocessing/Data Augmentation) ===")

for x_batch, y_batch in train_ds.take(1):
    print("Train batch x shape:", x_batch.shape)
    print("Train batch y shape:", y_batch.shape)

print("\n=== TEST DATASET after Preprocessing ===")

for x_batch, y_batch in test_ds.take(1):
    print("Test batch x shape:", x_batch.shape)
    print("Test batch y shape:", y_batch.shape)

train_size = tf.data.experimental.cardinality(train_ds).numpy() * BATCH_SIZE
test_size = tf.data.experimental.cardinality(test_ds).numpy() * BATCH_SIZE

print("\n=== DATASET SIZE CHECK ===")
print("Approx train samples:", train_size)
print("Approx test samples:", test_size)

#did preprocessing/augmentation work?
print("\n=== SAMPLE TRANSFORM CHECK ===")

for x_batch, y_batch in train_ds.take(1):
    print("Min pixel value:", tf.reduce_min(x_batch).numpy())
    print("Max pixel value:", tf.reduce_max(x_batch).numpy())
    print("Mean pixel value:", tf.reduce_mean(x_batch).numpy())



=== TRAIN DATASET after Prepeocessing/Data Augmentation) ===
Train batch x shape: (64, 3, 32, 32)
Train batch y shape: (64,)

=== TEST DATASET after Preprocessing ===
Test batch x shape: (64, 3, 32, 32)
Test batch y shape: (64,)

=== DATASET SIZE CHECK ===
Approx train samples: 45056
Approx test samples: 10048

=== SAMPLE TRANSFORM CHECK ===
Min pixel value: -1.0
Max pixel value: 1.0
Mean pixel value: 0.04237852


In [114]:
#Training

In [115]:
#Ordnerstrukur für Conatiner erstellen
container_path = f"containers/{CONTAINER_NAME}"
os.makedirs(f"{container_path}/cache", exist_ok=True)
os.makedirs(f"{container_path}/times", exist_ok=True)
os.makedirs(f"{container_path}/outputs", exist_ok=True)

In [116]:
#Initialisierung
init=subprocess.run([
    "python", "../sisa-fork/distribution.py",
    "--shards",str(NUM_SHARDS),
    "--container", CONTAINER_NAME,
    "--dataset", DATAFILEPATH,
    "--label", LABEL
],
capture_output=True,  # Fängt stdout und stderr ab
text=True             # Gibt die Ausgabe als Text zurück
)
print("STDOUT:", init.stdout)
print("STDERR:", init.stderr)
print("Returncode:", init.returncode)

STDOUT: 
STDERR: 
Returncode: 0


In [117]:
# Überprüfung splitfile.npy
splitfile_path = f"{container_path}/splitfile.npy"
if os.path.exists(splitfile_path):
    shards = np.load(splitfile_path, allow_pickle=True)
    print(f"\nInhalt von splitfile.npy (Anzahl der Shards: {len(shards)}):")
    for i, shard in enumerate(shards):
        print(f"  Shard {i}: {len(shard)} Samples")


Inhalt von splitfile.npy (Anzahl der Shards: 10):
  Shard 0: 5000 Samples
  Shard 1: 5000 Samples
  Shard 2: 5000 Samples
  Shard 3: 5000 Samples
  Shard 4: 5000 Samples
  Shard 5: 5000 Samples
  Shard 6: 5000 Samples
  Shard 7: 5000 Samples
  Shard 8: 5000 Samples
  Shard 9: 5000 Samples


In [15]:
#Jetzt 2 dateien erstellt
#splitfile.npy:  Eine Liste vonEine Liste von NumPy-Arrays, wobei jedes Array die Indizes der zu löschenden Daten für einen Shard enthält NumPy-Arrays, wobei jedes Array die Indizes der Trainingsdaten für einen Shard enthält
#requestfile:<label>.npy: Eine Liste von NumPy-Arrays, wobei jedes Array die Indizes der zu löschenden Daten für einen Shard enthält -> Hier Initiales Training,also Datei leer

In [74]:
# Training
#alle Arumente als Strings, da subprocess das so braucht
for i in range(NUM_SHARDS):
    print(f"Training Shard {i+1}/{NUM_SHARDS}...")
    train_all=subprocess.run([
        "python", "../sisa/sisa-fork/sisa.py",
        "--model", MODEL_NAME,
        "--train",
        "--slices", str(NUM_SLICES),
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--learning_rate", str(LEARNING_RATE),
        "--optimizer", OPTIMIZER,
        "--chkpt_interval", "1",
        "--container", CONTAINER_NAME,
        "--shard", str(i)
    ],
capture_output=True,  # Fängt stdout und stderr ab
text=True             # Gibt die Ausgabe als Text zurück
)
print(f"Ouput of Traing shard {i+1}/{NUM_SHARDS}")
print("STDOUT:", train_all.stdout)
print("STDERR:", train_all.stderr)
print("Returncode:", train_all.returncode)

Training Shard 1/10...
Training Shard 2/10...
Training Shard 3/10...
Training Shard 4/10...
Training Shard 5/10...
Training Shard 6/10...
Training Shard 7/10...
Training Shard 8/10...
Training Shard 9/10...
Training Shard 10/10...
Ouput of Traing shard 10/10
STDOUT: 
STDERR: C:\Users\sophi\miniconda3\envs\thesis\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Traceback (most recent call last):
  File "C:\Users\sophi\OneDrive\Documents\TH\Bachelorarbeit\Code\notebooks\sisa\sisa-fork\sisa.py", line 186, in <module>
    for images, labels in fetchShardBatch(
  File "C:\Users\sophi\OneDrive\Documents\TH\Bachelorarbeit\Code\notebooks\sisa\sisa-fork\sharded.py", line 48, in fetchShardBatch
    dataloader = importlib.import_module('.'.join(dataset.split('/')[:-1] + [datasetfile

In [14]:
X, y = load(np.arange(10), category='train')
print("X shape:", X.shape)  # Sollte (10, 3, 32, 32) sein
print("y shape:", y.shape)  # Sollte (10,) sein

X shape: (10, 3, 32, 32)
y shape: (10,)


In [119]:
#Prüfen ob Training funktioniert hat
result=subprocess.run([
    "python", r"../sisa-fork/sisa.py",  # Rohstring für den Pfad
     "--model", MODEL_NAME,
        "--train",
        "--slices", str(NUM_SLICES),
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--epochs", "1", #str(EPOCHS)
        "--batch_size", str(BATCH_SIZE),
        "--learning_rate", str(LEARNING_RATE),
        "--optimizer", OPTIMIZER,
        "--chkpt_interval", "1",
        "--container", CONTAINER_NAME,
        "--shard", "0"
],
capture_output=True,  # Fängt stdout und stderr ab
text=True             # Gibt die Ausgabe als Text zurück
)
# Gib die Standardausgabe und Fehlerausgabe aus
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Returncode:", result.returncode)

STDOUT: x_train shape: (45000, 32, 32, 3)
y_train shape: (45000, 1)
x_val shape: (5000, 32, 32, 3)
y_val shape: (5000, 1)
x_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000

STDERR: C:\Users\sophi\miniconda3\envs\thesis\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
I0000 00:00:1782664120.885527   12368 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782664126.354561   12368 port.cc:153] oneDNN custom operations are on. You may see slightly different nu

In [122]:
import torch

# Lade das gespeicherte Modell
checkpoint = torch.load(f"containers/{CONTAINER_NAME}/cache/shard-0-0.pt", map_location=torch.device('cpu'))
print("Gespeicherte conv1.weight Form:", checkpoint['model.conv1.weight'].shape)
# Sollte torch.Size([64, 3, 3, 3]) sein!

Gespeicherte conv1.weight Form: torch.Size([64, 3, 3, 3])


In [123]:
cache_dir = "containers/Baseline/cache"
if os.path.exists(cache_dir):
    print(f"Inhalt von {cache_dir}:")
    for file in os.listdir(cache_dir):
        file_path = os.path.join(cache_dir, file)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path)
            print(f"  - {file} ({size} Bytes)")
        else:
            print(f"  - {file}/ (Ordner)")
else:
    print(f"❌ Ordner {cache_dir} existiert nicht!")

Inhalt von containers/Baseline/cache:
  - 374d7dbee26e7eaadb7198c0907d72453a22938a6c83ea8ba4030ee168dd0e39.pt (44783371 Bytes)
  - 86dd7f9ed59332d148e0de9c4c33b5aac445c31b5336019bcf43842d533e0e35.pt (44783371 Bytes)
  - f01299897823b1f8546c5c4474ba85be48658ec45fd6d7f509cb7e81681e0688.pt (44783371 Bytes)
  - f01a1d65590a792a9a96af113c53e1f0c97b5a641915821be3c2b1b6752aa070.pt (44783371 Bytes)
  - f71d1ca4dc8985bddcd756d0ed72a34c818441a92a9bb7d432333f8a98450bdd.pt (44783371 Bytes)
  - shard-0-0.pt (44783371 Bytes)


In [ ]:
# Vorhersagen
for i in range(NUM_SHARDS):
    print(f"Predicting with Shard {i+1}/{NUM_SHARDS}...")
    prediction_all=subprocess.run([
        "python", "../sisa-fork/sisa.py",
        "--model", MODEL_NAME,
        "--test", #Important
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--batch_size", str(BATCH_SIZE),
        "--container", CONTAINER_NAME,
        "--shard", str(i)
    ], 
capture_output=True, 
text=True)
print("STDOUT:", prediction_all.stdout)
print("STDERR:", prediction_all.stderr)
print("Returncode:", prediction_all.returncode)

In [124]:
#Prüfen ob Vorhersagen funktioniert
predict=subprocess.run([
    "python", "../sisa-fork/sisa.py",
    "--model", MODEL_NAME,
    "--test", #Important
    "--dataset", DATAFILEPATH,
    "--label", LABEL,
    "--batch_size", str(BATCH_SIZE),
    "--container", CONTAINER_NAME,
    "--shard", "0"
    ],
capture_output=True,  # Fängt stdout und stderr ab
text=True             # Gibt die Ausgabe als Text zurück
)
# Gib die Standardausgabe und Fehlerausgabe aus
print("STDOUT:", predict.stdout)
print("STDERR:", predict.stderr)
print("Returncode:", predict.returncode)

STDOUT: x_train shape: (45000, 32, 32, 3)
y_train shape: (45000, 1)
x_val shape: (5000, 32, 32, 3)
y_val shape: (5000, 1)
x_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000

STDERR: C:\Users\sophi\miniconda3\envs\thesis\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
I0000 00:00:1782664334.167436   18404 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782664338.576745   18404 port.cc:153] oneDNN custom operations are on. You may see slightly different nu

In [125]:
#7. Aggregation 

aggregation = subprocess.run([
    "python", "../sisa-fork/aggregation.py",
    "--strategy", "uniform",
    "--container", CONTAINER_NAME,
    "--shards", "1", #str(NUM_SHARDS)
    "--dataset", DATAFILEPATH,
    "--label", LABEL
], 
capture_output=True, 
text=True)
print("STDOUT:", aggregation.stdout)
print("STDERR:", aggregation.stderr)
print("Returncode:", aggregation.returncode)

STDOUT: x_train shape: (45000, 32, 32, 3)
y_train shape: (45000, 1)
x_val shape: (5000, 32, 32, 3)
y_val shape: (5000, 1)
x_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000
0.3532

STDERR: WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1782664421.967937   29352 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782664426.385730   29352 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
C:\Users\sophi\miniconda3\envs\thesis\Lib\site-

In [30]:
# Lade Vorhersagen aller Shards
outputs = []
for i in range(1): #NUM_SHARDS
    pred_file = f"../sisa-fork/containers/{CONTAINER_NAME}/outputs/shard-{i}-{LABEL}.npy"
    pred = np.load(pred_file)
    outputs.append(pred)
outputs = np.array(outputs)

In [32]:
# Lade die Vorhersagen für Shard 0
predictions = np.load("containers/Baseline/outputs/shard-0-0.npy")
print("Form der Vorhersagen:", predictions.shape)  # Sollte (10000, 10) sein (10000 Testbilder, 10 Klassen)
print("Beispiel-Vorhersagen (erste 5):\n", predictions[:5])  # Sollte Wahrscheinlichkeiten oder Logits sein

Form der Vorhersagen: (10000, 10)
Beispiel-Vorhersagen (erste 5):
 [[0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]]


In [ ]:
#TODO: get the general-report.csv for results 

In [ ]:
# Unlearning

In [126]:
#Requests manuell vorbereiten
# Pfad zur requestfile
requestfile_path = f"containers/{CONTAINER_NAME}/requestfile-{LABEL}.npy"

# Erstelle die requestfile basierend Dateien -> Loeschszenarios
forget_indices = np.load(r"C:\Users\sophi\OneDrive\Documents\TH\Bachelorarbeit\Code\notebooks\data\cifar10_split10\x_delete10.npy")  # Indizes der zu löschenden Datenpunkte
NUM_SHARDS = 10  

# Berechne die Anzahl der zu löschenden Datenpunkte pro Shard
forget_per_shard = len(forget_indices) // NUM_SHARDS
requests = np.full((NUM_SHARDS,), forget_per_shard, dtype=int)

# Speichere die requestfile
os.makedirs(os.path.dirname(requestfile_path), exist_ok=True)
np.save(requestfile_path, requests)

print(f"✅ Unlearning-Request direkt erstellt: {requestfile_path}")
print(f"Inhalt: {requests}")

✅ Unlearning-Request direkt erstellt: containers/Baseline/requestfile-0.npy
Inhalt: [500 500 500 500 500 500 500 500 500 500]


In [127]:
# Überprüfung requestfile-0.npy
requestfile_path = f"{container_path}/requestfile-0.npy"
if os.path.exists(requestfile_path):
    requests = np.load(requestfile_path, allow_pickle=True)
    print(f"\nInhalt von requestfile:0.npy (Anzahl der Shards: {len(requests)}):")
    for i, req in enumerate(requests):
        print(f"  Shard {i}: {req} Requests")


Inhalt von requestfile:0.npy (Anzahl der Shards: 10):
  Shard 0: 500 Requests
  Shard 1: 500 Requests
  Shard 2: 500 Requests
  Shard 3: 500 Requests
  Shard 4: 500 Requests
  Shard 5: 500 Requests
  Shard 6: 500 Requests
  Shard 7: 500 Requests
  Shard 8: 500 Requests
  Shard 9: 500 Requests


In [65]:
# Generiere Requests
#Ich habe schon vorbereitete Datensaetze -> Basis der Requests
#TODO: weg finden, datei direkt zu nehmen
#Kann nicht benutzt werden, da custom Datenpunkte nicht möglich


#unlearning_requests=subprocess.run([
#    "python", "../sisa-fork/distribution.py",
#    "--requests", 




In [50]:
# Trainiere betroffenen Shards neu
#Hier grade alle Shards aber das schlecht
for i in range(shards):
    print(f"Retraining Shard {i+1}/{shards} with Unlearning...")
    retrain_all=subprocess.run([
        "python", "../sisa-fork/sisa.py",
        "--model", MODEL_NAME,
        "--train",
        "--slices", "1",
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--learning_rate", str(LEARNING_RATE),
        "--optimizer", OPTIMIZER,
        "--chkpt_interval", "1",
        "--container", CONTAINER_NAME,
        "--shard", str(i)
    ], 
capture_output=True, 
text=True)
print("STDOUT:", retrain_all.stdout)
print("STDERR:", retrain_all.stderr)
print("Returncode:", retrain_all.returncode)


TypeError: only integer scalar arrays can be converted to a scalar index

In [ ]:
#TODO: symfile loeschen

In [129]:
#Prüfen ob Training funktioniert hat
result=subprocess.run([
    "python", r"../sisa-fork/sisa.py",  # Rohstring für den Pfad
     "--model", MODEL_NAME,
        "--train",
        "--slices", str(NUM_SLICES),
        "--dataset",  DATAFILEPATH,
        "--label", LABEL,
        "--epochs", "1", #str(EPOCHS)
        "--batch_size", str(BATCH_SIZE),
        "--learning_rate", str(LEARNING_RATE),
        "--optimizer", OPTIMIZER,
        "--chkpt_interval", "1",
        "--container", CONTAINER_NAME,
        "--shard", "0"
],
capture_output=True,  # Fängt stdout und stderr ab
text=True             # Gibt die Ausgabe als Text zurück
)
# Gib die Standardausgabe und Fehlerausgabe aus
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Returncode:", result.returncode)

STDOUT: 
STDERR: C:\Users\sophi\miniconda3\envs\thesis\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)

Returncode: 0


In [104]:
# Berechne Vorhersagen für Unlearning
for i in range(shards):
    prediction_unlearning=subprocess.run([
        "python", "../sisa-fork/sisa.py",
        "--model",MODEL_NAME,
        "--test",
        "--dataset", DATAFILEPATH,
        "--label", LABEL,
        "--batch_size", str(BATCH_SIZE),
        "--container", CONTAINER_NAME,
        "--shard", str(i)
    ], 
capture_output=True, 
text=True)
print("STDOUT:", prediction_unlearning.stdout)
print("STDERR:", prediction_unlearning.stderr)
print("Returncode:", prediction_unlearning.returncode)

TypeError: only integer scalar arrays can be converted to a scalar index

In [130]:
predict=subprocess.run([
    "python", "../sisa-fork/sisa.py",
    "--model", MODEL_NAME,
    "--test", #Important
    "--dataset", DATAFILEPATH,
    "--label", LABEL,
    "--batch_size", str(BATCH_SIZE),
    "--container", CONTAINER_NAME,
    "--shard", "0"
    ],
capture_output=True,  # Fängt stdout und stderr ab
text=True             # Gibt die Ausgabe als Text zurück
)
# Gib die Standardausgabe und Fehlerausgabe aus
print("STDOUT:", predict.stdout)
print("STDERR:", predict.stderr)
print("Returncode:", predict.returncode)

STDOUT: x_train shape: (45000, 32, 32, 3)
y_train shape: (45000, 1)
x_val shape: (5000, 32, 32, 3)
y_val shape: (5000, 1)
x_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000

STDERR: C:\Users\sophi\miniconda3\envs\thesis\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
I0000 00:00:1782664636.175109   25284 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782664640.543718   25284 port.cc:153] oneDNN custom operations are on. You may see slightly different nu

In [131]:
# Aggregiere Vorhersagen

aggregation_unlearn = subprocess.run([
    "python", "../sisa-fork/aggregation.py",
    "--strategy", "uniform",
    "--container", CONTAINER_NAME,
    "--shards", "1", #str(NUM_SHARDS)
    "--dataset", DATAFILEPATH,
    "--label",  LABEL
], 
capture_output=True, 
text=True)
print("STDOUT:", aggregation_unlearn.stdout)
print("STDERR:", aggregation_unlearn.stderr)
print("Returncode:", aggregation_unlearn.returncode)




STDOUT: x_train shape: (45000, 32, 32, 3)
y_train shape: (45000, 1)
x_val shape: (5000, 32, 32, 3)
y_val shape: (5000, 1)
x_test shape: (10000, 32, 32, 3)
y_test shape: (10000, 1)
Number of training datapoints: 45000
Number of validation datapoints: 5000
Number of test datapoints: 10000
0.3566

STDERR: WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1782664733.692957   30820 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782664738.301572   30820 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
C:\Users\sophi\miniconda3\envs\thesis\Lib\site-